# Quantum Chemistry with ORCA
## Introduction

In this notebook, we will carry out quantum chemical calculations on organic and inorganic molecules. 
- We will use the ORCA program package developed at the Max-Planck-Institut für Kohlenforschung.
- ORCA is a general-purpose quantum chemistry package initiated by Frank Neese in 1999.
- ORCA is free for academic use and features a variety of quantum chemical methods.
- We will be using Density Functional Theory (DFT) and Gaussian-Type basis sets.

## Learning outcomes

After completing this notebook, you will be able to:
- Understand the key parts of ORCA input files for molecular systems.
- Optimize molecular structures with ORCA.
- Compare total energies of molecular isomers to find out the lowest-energy isomer.
- Calculate and interpret infrared vibrational spectra for molecules.

## Instructions for using the notebook
- The notebook is composed of cells and the blue highlight in the left side of the notebook shows which cell is currently active. 
- You can run the active cell by clicking the "play" button from the toolbar above ("Run this cell and advance", keyboard shortcut is Shift + Enter).
- The notebook includes instruction cells like this cell and gray cells that contain Python code.
- When you run a cell and see the symbol `In [*]` in the top left corner of the cell, the notebook is executing code. Please wait until the symbol changes to a number like `[1]`.
- Run the cells from the top to the bottom.
- The files generated during the exercise are visible in the file browser on the left side of the notebook. Text files (.inp, .xyz, and .out extension) can be opened for viewing by double-clicking them.
- If something goes totally wrong, you can open the Kernel menu and choose "Restart Kernel and Clear Outputs of All Cells". This will reset the state of the notebook and you can start from the beginning.

## Setting up the calculation environment for ORCA

Run the next code block to setup the calculation environment for ORCA.

In [ ]:
import sys
sys.path.append('../tools')
from qctools import load_xyz, save_xyz, load_molecule_pubchem, load_xyz_as_traj, show_molecule

## ORCA walkthrough for a simple molecule: H<sub>2</sub>O
Let's get familiar with ORCA by using a gas-phase H<sub>2</sub>O molecule as an example.

### Creating subdirectories
ORCA creates a large number of files during the execution. In this notebook, all ORCA input and output files are stored inside subdirectories. 

Run the following code block to create a subdirectory `h2o` for our first job.

In [ ]:
!mkdir h2o

`!mkdir` means that the notebook calls the Unix shell command `mkdir` to create a directory. We could also create directories with Python but in this case it is more convenient to work directly with shell commands.

> The newly created subdirectory `h2o` should soon appear in the file browser on the left.

### Writing files and visualizing molecular structures

Let's start by visualizing a molecular structure using NGLview tool. We first need to create a molecular structure file. 
- We can write files in notebooks with the `%%writefile` command (a so-called IPython magic command available in notebooks).
- `%%writefile h2o/h2o_dist.xyz` means that we will create a file called `h2o_dist.xyz` in the subdirectory `h2o` that we just created above.
- The file format is so-called XYZ file. This is a standard human-readable file format in quantum chemistry. It contains the cartesian coordinates of each atom in format `symbol X Y Z`).

Run the following code block to create an XYZ file with a distorted water molecular (H-O-H angle of 60 degrees, O-H distances of 1.2 Å).

In [ ]:
%%writefile h2o/h2o_dist.xyz
3
Distorted water molecule
 O                  0.00000000    0.00000000   -0.16627688
 H                  0.00000000   -0.60000000    0.87295361
 H                 -0.00000000    0.60000000    0.87295361

> You can use the file browser on the left to check that the file `h2o_dist.xyz` indeed appeared inside the `h2o` subdirectory (double-click the folder to open it). You can return to the original working directory by clicking `basic` in the directory path `/ ... / basic / h2o` in the top of the file browser.

Next, run the following code block to load the distorted water molecule in an NGLview window.

In [ ]:
mol = load_xyz('h2o/h2o_dist.xyz')
show_molecule(mol)

### Interacting with the molecular structure
You can interact with the molecular structure in the above view as follows:
- Rotate: hold the left mouse button and move the mouse.
- Zoom: mouse scrollwheel.
- Translate: hold the right mouse button and move the mouse.
  
Next, we will optimize the distorted structure to a reasonable minimum. 

### Structure optimization of H<sub>2</sub>O

In quantum chemical molecular modelling, the first task is usually the structural optimization of the studied system. This means that we will find the lowest-energy three-dimensional structure for the molecule.

Standard structure optimization methods can find local minima on the potential energy surface. This means that the optimization algorithm does not search all possible conformations of the molecule. We will have a look into conformation analysis later.

Let's optimize the structure of water molecule with Density Functional Theory (DFT). We will use the DFT-PBE functional, which is the most commonly used GGA-level DFT functional. We will look into more accurate DFT methods later, but for typical main group compounds DFT-PBE performs reasonably well. 

Let's start by writing an ORCA input file. The input file shown in the code block below consists of following items:
- The first line in the code block is the `%%writefile` command that writes the following lines into a file `h2o_opt.inp` located in subdirectory `h2o` created above. The convention is that ORCA input files have file extension `.inp`.
- The line starting with `!` is so-called ORCA simple input line. Here we define the DFT-PBE method (`PBE`) and the one-electron basis set (`DEF2-TZVP`). The `Opt` keyword means that we want to optimize the molecular structure.
- The simple input line is followed by the XYZ coordinates of the water molecule. The line `xyz 0 1` tells ORCA that we will give the molecular structure in XYZ format, the charge of the molecule is 0 and the spin multiplicity is 1 (singlet, no unpaired electrons).
- The calculation will use only one CPU core because the system is so small. Later, we will add more CPUs (computing power).

Run the next code block to create the input file.

In [ ]:
%%writefile h2o/h2o_opt.inp
!PBE DEF2-TZVP Opt
* xyz 0 1
 O                  0.00000000    0.00000000   -0.16627688
 H                  0.00000000   -0.60000000    0.87295361
 H                 -0.00000000    0.60000000    0.87295361
*

On a general level, the structural optimization will proceed as follows:

- ORCA calculates the electronic structure (molecular orbitals) of the initial structure. This also yields the electronic total energy of the system.
- ORCA calculates the forces affecting the atoms (first derivatives of the total energy with respect to atomic coordinates).
- ORCA displaces the atoms in such way that the forces affecting the atoms are reduced. The goal is to reach a local minimum structure where all forces are zero (in practice, close to zero).
- The above three steps are repeated until the forces affecting the atoms are below the default convergence criteria in ORCA. These are relatively strict, and can be changed by the user.
- After the convergence criteria have been reached, the structure optimization is complete.

Run the next code block to run the structure optimization. The job should take less than 30 seconds to finish.

In [ ]:
!cd h2o ; $ORCA_HOME/orca h2o_opt.inp > h2o_opt.out
!grep -E "TERMINATED|TOTAL RUN TIME" h2o/h2o_opt.out

Let's load all intermediate structures of the optimization to see how it proceeded (this is often called the "trajectory"). 

Run the following code block, which loads the optimization trajectory. After loading, the view shows the initial structure. Move your mouse  over the view and press the ▶ button to see how the optimization changed the molecular structure.

In [ ]:
mol2 = load_xyz_as_traj('h2o/h2o_opt_trj.xyz')
show_molecule(mol2)

### Energy changes during the structural optimization of H<sub>2</sub>O

Let's see how the total electronic energy of the system changed during the optimization. The next code block will read the total electronic energies and plot them with Matplotlib. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Extract the electronic total energies from the ORCA output
energies_raw = !grep "FINAL SINGLE POINT ENERGY" h2o/h2o_opt.out | awk '{print $NF}'

energies = np.asarray(energies_raw, dtype=float)
steps = np.arange(1, len(energies_raw) + 1)

plt.plot(steps, energies, marker = 'o', color = 'black', label = 'Total energy')
plt.xlabel('Optimization step')
plt.ylabel('Total energy (Hartree)')
plt.xlim(1, len(steps) + 1)
plt.ylim(np.min(energies)*1.0001, np.max(energies))
plt.xticks(np.arange(1, len(steps) + 1, 1))
plt.yticks(np.linspace(np.min(energies), max(energies), 6))
plt.legend(loc = 'upper right')
plt.title('Optimization of H$_2$O')
plt.show()

The plot above shows how the total energy decreases and then no longer changes. In chemistry, we are usually more interested in relative energies. Let's see how the same plot looks when we set the energy of the optimized structure as zero and calculate the relative energies of the intermediate structures.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Extract the electronic total energies from the ORCA output
energies_raw = !grep "FINAL SINGLE POINT ENERGY" h2o/h2o_opt.out | awk '{print $NF}'

# The total energies in ORCA are in Hartree units. 
# We use this conversion factor to convert to kJ/mol
HARTREE_TO_KJMOL = 2625.5
energies = np.asarray(energies_raw, dtype=float)
rel_energies = (energies - np.min(energies)) * HARTREE_TO_KJMOL
steps = np.arange(1, len(energies_raw) + 1)

plt.plot(steps, rel_energies, marker = 'o', color = 'red', label = 'Relative energy')
plt.xlabel('Optimization step')
plt.ylabel('Relative energy (kJ/mol)')
plt.xlim(1, len(steps) + 1)
plt.ylim(0, np.max(rel_energies))
plt.xticks(np.arange(1, len(steps) + 1, 1))
plt.yticks(np.linspace(np.min(rel_energies), max(rel_energies), 6))
plt.legend(loc = 'upper right')
plt.title('Optimization of H$_2$O')
plt.show()

The plot above shows that the starting structure was indeed a high-energy structure (224 kJ/mol higher in energy compared to the final minimum).

### Harmonic frequency calculation for H<sub>2</sub>O

The structure optimization of the water molecule finished because all forces affecting the atoms are close to zero. This means that we have reached a stationary point at the potential energy surface.

Next, we can run a so-called frequency calculation to check that we have reached a true local minimum. The frequencies are typically calculated within the harmonic approximation. This means that the vibrational modes are described as harmonic oscillators that are independent of each other. It is also possible to study anharmonic frequencies where the vibrational modes are coupled to each other but this is out of the scope of the current tutorial.

In a frequency calculation, we are calculating the second derivatives of the total energy with respect to atomic coordinates. This yields so-called Hessian matrix (Hessian) which describes the local curvature of the potential energy surface:
- If all eigenvalues of the Hessian matrix are positive, structure is a true local minimum. All vibrational frequencies are positive.
- If the Hessian has one negative eigenvalue, the structure is a transition state. One vibrational frequency has an imaginary value (often denoted as negative frequency due to technical reasons)
- If the Hessian has several negative eigenvalues, the structure is a saddle point. There are several imaginary vibrational frequencies.

Let's write an ORCA input file for the frequency calculation. The input file shown in the next code block has some changes compared to the structure optimization input:
- The keyword `Opt` has been replaced with keyword `Freq`.
- The line `xyzfile 0 1 h2o/h2o_opt.xyz` tells ORCA to read the optimized structure from the file `h2o/h2o_opt.xyz`. Charge is 0 and spin multiplicity is 1 similar to the optimization input above.
>We need to include the directory `h2o` in the path because the ORCA binary will run in the main working directory `basic`, not in the subdirectory `h2o`.

Run the next code block to write the input file for the frequency calculation.

In [ ]:
%%writefile h2o/h2o_fq.inp
!PBE DEF2-TZVP Freq
* xyzfile 0 1 h2o_opt.xyz

Run the next code block to run the frequency calculation. The job should take less than 30 seconds to finish.

In [ ]:
!cd h2o ; $ORCA_HOME/orca h2o_fq.inp > h2o_fq.out
!grep -E "TERMINATED|TOTAL RUN TIME" h2o/h2o_fq.out

The next code block will extract the calculated vibrational frequencies from the output file using the `sed` utility.

In [ ]:
!sed -n -e '/VIBRATIONAL FREQUENCIES/,/NORMAL MODES/p' h2o/h2o_fq.out | head -n -3

There are in principle 9 vibrational frequencies listed (note that indexing starts from 0). However, the first six models are zero frequencies that correspond to translations and rotations. For a non-linear molecule with *N* atoms, there are 3*N* nonzero vibrational modes. 

### Visualizing the harmonic frequencies of H<sub>2</sub>O

We can plot the three nonzero modes in `xyz` trajectory format with the `orca_pltvib` command:

In [ ]:
!cd h2o ; orca_pltvib h2o_fq.hess 6 7 8

After this, the trajectory files located in the `h2o` subdirectory can be visualized to show the vibrational frequencies.

After running the next code block, move your mouse over the view, and press the third button in the toolbar that appears ("repeat"). Then, press play to see the vibrational frequency.

In [ ]:
# Visualize the vibrational frequency at 1583.29 cm**-1
freq = load_xyz_as_traj('h2o/h2o_fq.hess.v006.xyz')
show_molecule(freq)

> #### Exercise
> Modify the **trajectory file name** in the previous previous code and run the code block to visualize the two other vibrational frequencies of water.

### Infrared spectrum of H<sub>2</sub>O in the gas phase

Finally, we can plot the IR spectrum of water molecule in the gas phase. IR intensity of a vibrational mode depends on the change of dipole moment during the vibration. ORCA calculates the IR intensities by default during a harmonic frequency calculation.

First, generate the IR spectrum plot with the `orca_mapspc` tool:

In [ ]:
!cd h2o ; orca_mapspc h2o_fq.hess IR -w32

Parameter `-w32` determines the linewidth (Gaussian shape, Full width at half maximum = 32 cm<sup>-1</sup>). Next, plot the spectrum.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load the raw data
ir_data = np.loadtxt('h2o/h2o_fq.hess.ir.dat')

frequencies = ir_data[:, 0] # Wavenumbers in the first column
transmittances = (ir_data[:, 1] - 999) * 100 # Transmittances in the second column
plt.plot(frequencies, transmittances, '-', color = 'red', label = 'IR')
plt.xlabel('Frequency (cm$^{-1}$)')
plt.ylabel('Transmittance (%)')
plt.xlim(max(frequencies), min(frequencies), )
plt.ylim(min(transmittances) * 0.999, max(transmittances) * 1.001)
plt.xticks(np.linspace(500, 4000, 8))
plt.yticks(np.linspace(min(transmittances), max(transmittances), 10))
plt.title('IR spectrum of H$_2$O')
plt.show()

Final note: It is possible to include both `Opt` and `Freq` keywords in the same input. In this case, Orca first optimizes the strucure and then calculates the harmonic frequencies.

## Complete workflow for an inorganic 5*d* metal compound: WF<sub>6</sub>

Tungsten hexafluoride WF<sub>6</sub> is used as a precursor molecule to form tungsten thin-films in semiconductor industry. Let's study the structure and vibrational properties of WF<sub>6</sub>.

Let's first create a new subdirectory `wf6`:

In [ ]:
!mkdir wf6

When investigating *d*-metals, it is important to pay attention on the DFT exchange-correlation (XC) functional. The DFT-PBE method we used above for water may not be accurate enough. In this example, we will be using the hybrid **DFT-PBE0** functional which includes 25% of exact exchange (Hartree-Fock exchange) in the XC functional. We will be using the same def2-TZVP basis set as in the example above. However, in this case, 60 core electrons of tungsten are replaced by an effective core potential (ECP). This leaves 14 electrons on each tungsten atom. The ECP takes care of scalar relativistic effects and also decreases the computational cost as 60 less electrons need to be treated explicitly.

### Structural optimization and frequency calculation of WF<sub>6</sub>
The first step is to write the input file. Usually it is convenient to carry out both structure optimization and harmonic frequency calculation in the same job. This can be done by including both `Opt` and `Freq` keywords.

In [ ]:
%%writefile wf6/wf6.inp
!PBE0 DEF2-TZVP Opt Freq
!PAL4
* xyz 0 1
 W                 -0.13573497    0.20359664   -0.00355389
 F                 -0.13573497    0.20359664    1.87644611
 F                  1.74426503    0.20359664   -0.00355389
 F                 -2.01573497    0.20359664   -0.00355389
 F                 -0.13573497    0.20359664   -1.88355389
 F                 -0.13573497    2.08359664   -0.00355389
 F                 -0.13573497   -1.67640336   -0.00355389
*

Run the job (will take about 6 to 8 minutes):

In [ ]:
!cd wf6 ; $ORCA_HOME/orca wf6.inp > wf6.out
!grep -E "TERMINATED|TOTAL RUN TIME" wf6/wf6.out

Let's see how the electronic energy energy changed during the optimization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

energies_raw = !grep "FINAL SINGLE POINT ENERGY" wf6/wf6.out | awk '{print $NF}'

HARTREE_TO_KJMOL = 2625.5
energies = np.asarray(energies_raw, dtype=float)
rel_energies = (energies - np.min(energies)) * HARTREE_TO_KJMOL
steps = np.arange(1, len(energies_raw) + 1)

plt.plot(steps, rel_energies, marker = 'o', color = 'red', label = 'Relative energy')
plt.xlabel('Optimization step')
plt.ylabel('Relative energy (kJ/mol)')
plt.xlim(1, len(steps) + 1)
plt.ylim(0, np.max(rel_energies))
plt.xticks(np.arange(1, len(steps) + 1, 1))
plt.yticks(np.linspace(np.min(rel_energies), max(rel_energies), 6))
plt.gca().yaxis.set_major_formatter(StrMethodFormatter('{x:.1f}'))
plt.legend(loc = 'upper right')
plt.title('Optimization of WF$_6$')
plt.show()

Let's visualize the optimization trajectory

In [ ]:
mol = load_xyz_as_traj('wf6/wf6_trj.xyz')
show_molecule(mol)

Extract the calculated vibrational frequencies from the output file:

In [ ]:
!sed -n -e '/VIBRATIONAL FREQUENCIES/,/NORMAL MODES/p' wf6/wf6.out | head -n -3

> #### Question
>
> Did the structure optimization reach a true local minimum?

Plot all vibrational modes in `xyz` trajectory format with the `orca_pltvib` command:

In [ ]:
!cd wf6 ; orca_pltvib wf6.hess all

The vibrational frequencies can be visualized as in the example above. Let's first visualize the lowest-frequency vibrational mode 6:

In [ ]:
freq = load_xyz_as_traj('wf6/wf6.hess.v006.xyz')
show_molecule(freq)

> #### Exercise
> What is the wavenumber (frequency) of the fully symmetric W–F stretching vibration (all W–F bonds are stretching in the same phase)? Change the name of the trajectory file name in the code block above and visualize the vibrations to find the answer.

### Molecular orbitals of WF<sub>6</sub>

 As a result of the DFT-PBE0 calculation, we have obtained the molecular orbitals (MOs) of WF<sub>6</sub> (strictly speaking, we obtained the Kohn-Sham orbitals because we are using Kohn-Sham DFT, but in practice we can call them MOs).

Run the next code block to print all molecular orbitals of WF<sub>6</sub>. ORCA prints the molecular orbitals during the first step of the structural optimization and after the optimization has converged. If you are ever looking for molecular orbitals energies from an optimizatio output, be sure to take the values in the end of the file (after `FINAL ENERGY EVALUATION`).

In [ ]:
!sed -n -e '/FINAL ENERGY EVALUATION/,/Only the first 10/p' wf6/wf6.out | sed -n -e '/ORBITAL ENERGIES/,/Only the first 10/p'

The indexing of the MOs starts from 0. The orbitals with OCC = 2 are occupied while those with OCC = 0 are virtual (empty) orbitals. The highest occupied MO (HOMO) is orbital 33. Because the indexing of the orbitals start from zero, we have (33 + 1) * 2 = 68 electrons. We can double check this by printing the number of electrons from the ORCA output: 

In [ ]:
!grep -m 1 "Number of Electrons" wf6/wf6.out

MOs can be plotted with command `orca_plot`. The command reads the wavefunction from the file `wf6.gbw`. It is possible to run `orca_plot` interactively with `orca_plot gbwfile -i`. However, we will run the command with an input file that lists the orbitals that we want to plot.

We will be writing an input file like this (comments added below)
```
%%writefile wf6/moplot
2   -> Tell orca_plot to read MO number
33  -> MO number
11  -> Plot the MO (writes a so-called cube file)
12  -> Exit orca_plot
```
Run the next code block to save an input that generates cube files for MOs 33, 30, and 21.

In [ ]:
%%writefile wf6/moplot
2
33
11
2
30
11
2
21
11
12

Run `orca_plot`:

In [ ]:
!cd wf6 ; orca_plot wf6.gbw -i < moplot > /dev/null

Visualize the three MOs (33, 30, 21) one at a time by setting the variable `CUBEFILE` in the following code block and run it. The molecule is rather far when it loads, so you need to zoom in (with mouse wheel).

In [ ]:
# Set cube that you want to plot
CUBEFILE = "wf6/wf6.mo33a.cube"
# You can adjust isolevel between 0.01 and 0.08 to see how the extent of the orbital changes.
# The larger the number, the tighter the isosurface value cutoff for plotting.
ISOLEVEL = 0.02
# ----- No need to change anything below -----
mol = load_xyz('wf6/wf6.xyz')
view = show_molecule(mol)
view.add_component(CUBEFILE)
view.component_1.clear()
view.component_1.add_surface(opacity=0.5, color='blue', isolevelType="value", isolevel=ISOLEVEL)
view.component_1.add_surface(opacity=0.5, color='red',  isolevelType="value", isolevel=-ISOLEVEL)
view

The MO plots illustrate that these so-called canonical MOs are typically rather delocalised. It is possible to localize the molecular orbitals to get localized orbitals that are typically easier to connect to chemical bonding concepts typically discussed in textbooks.

## Structural isomers of butanol

[Butanoli](https://fi.wikipedia.org/wiki/Butanoli) (C<sub>4</sub>H<sub>9</sub>OH) on alkoholi, jolla on neljä erilaista rakenneisomeeriä: 1-butanoli, 2-butanoli, isobutanoli ja tert-butanoli.

Tehtävänäsi on selvittää rakenneisomeerien energiajärjestys, eli millä isomeerillä on matalin kokonaisenergia ja millä korkein.

### 1-butanoli

Selvitä Wikipedian ja PubChemin avulla, millä nimellä löydät 1-butanolin PubChemistä. Lataa rakenne ja visualisoi se.

In [ ]:
# Fill in the correct name
name = '1-butanol'
mol_1_butanol = load_molecule_pubchem(name)
show_molecule(mol_1_butanol)

Create input:

In [ ]:
%%writefile isomer_1-butanol.inp
!PBE DEF2-TZVP Opt
* xyzfile 0 1 isomer_1-butanol.xyz

Optimize the molecule:

In [ ]:
!$ORCA_HOME/orca isomer_1-butanol.inp > isomer_1-butanol.out
!grep "TERMINATED" isomer_1-butanol.out
!grep "TOTAL RUN TIME" isomer_1-butanol.out

### 2-butanoli

Selvitä Wikipedian ja PubChemin avulla, millä nimellä löydät 2-butanolin PubChemistä. Lataa rakenne ja visualisoi se.

In [ ]:
# TÄYTÄ TÄHÄN OIKEA NIMI
nimi = 'NIMI'
mol_2_butanoli = lataa_molekyyli_pubchem(nimi)
# Save as isomer_2-butanol
näytä_molekyyli(mol_2_butanoli)

### Rakenneisomeerien energiavertailu

Optimoidujen rakenteiden kokonaisenergioiden avulla voimme laskea rakenneisomeerien suhteelliset energiat. Täydennä seuraavaan koodiin kokonaisenergiat ja aja koodi:

In [ ]:
# Fill in total energies (in Hartree units)
isomers = ["1-butanol", "2-butanol", "isobutanol", "tert-butanol"]
# Replace the zeros in the list with total energies in the same order as in the above list isomers
energiat = [0, 0, 0, 0] 
###################################
# Muuntokerroin
muuntokerroin = 96.485 # eV -> kJ/mol
OK = True
for energia in energiat:
    if energia == 0:
        print("Muista täydentää kaikki kokonaisenergiat energiat-listaan!")
        OK = False
        break
if OK:        
    E_matalin = min(energiat)
    matalin = isomeerit[energiat.index(E_matalin)]
    print(f'Matalin kokonaisenergia {E_matalin} eV on isomeerilla {matalin}\n')

    print('Isomeerien suhteelliset energiat:')
    for isomeeri, energia in zip(isomeerit, energiat):
        E_suhteellinen = (energia - E_matalin) * muuntokerroin
        print(f'{isomeeri:13s} {E_suhteellinen:3.0f} kJ/mol')

Tulosten osalta on hyvä huomioida, että tämä vertailu pätee yksittäisille molekyyleille 0 K lämpötilassa. Vertailu huoneenlämpötilassa vaatisi tarkemman termodynaamisten suureiden käsittelyn ja liuostilan mallinnusta.

> #### Question
>
> Which structural isomer has the lowest energy? Why?

## Conformational analysis

With GOAT

## GGA vs. hybrid

Magnetic d-metal complex?

## Note about point group symmetry

Opt + Freq for SF<sub>6</sub> in *C*<sub>1</sub> vs. *O<sub>h</sub>*

## Running ORCA on CSC batch system

Orca kaksi työtä yhtä aikaa?